к сожалению, выводы ячеек с google collab не сохранились, для демонстрации результата без перезапуска ноутбука

In [ ]:
%pip install -q --upgrade \
  "transformers==4.53.3" "trl==0.19.1" "peft==0.16.0" \
  "accelerate==1.8.1" "bitsandbytes==0.46.1" "datasets==3.6.0" \
  "scikit-learn==1.7.2" "scipy==1.15.3" "safetensors==0.5.3"

In [ ]:
import gc
import json
import os
import pickle
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import requests
import torch
from datasets import Dataset
from scipy.sparse import hstack
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, PeftModel, TaskType, prepare_model_for_kbit_training
from trl import (
    CPOConfig,
    CPOTrainer,
    DPOConfig,
    DPOTrainer,
    RewardConfig,
    RewardTrainer,
    SFTConfig,
    SFTTrainer,
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False

print("seed:", SEED)

In [ ]:
DATA_URL = "https://edu.tbank.ru/files/c1005bf0-8695-451a-9616-87c8687dce27"
WORK = Path.cwd()
ZIP_PATH = WORK / "ml-olympiad-red-task.zip"
EXTRACTED = WORK
RUNS = WORK / "runs"

if not ZIP_PATH.exists():
    with requests.get(DATA_URL, stream=True, timeout=120) as response:
        response.raise_for_status()
        with ZIP_PATH.open("wb") as out:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    out.write(chunk)

if not (EXTRACTED / "ml-olympiad-red-task").exists():
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(EXTRACTED)

matches = list(EXTRACTED.rglob("kid_adult.jsonl"))
TASK_ROOT = matches[0].parent.parent
DATA_DIR = TASK_ROOT / "data"
METRICS_DIR = TASK_ROOT / "metrics"

if RUNS.exists():
    shutil.rmtree(RUNS)
RUNS.mkdir(parents=True)


In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

kid_adult = read_jsonl(DATA_DIR / "kid_adult.jsonl")
good_bad = read_jsonl(DATA_DIR / "good_bad.jsonl")
public_style = read_jsonl(DATA_DIR / "public_test_style.jsonl")
public_quality = read_jsonl(DATA_DIR / "public_test_quality.jsonl")

assert len(kid_adult) == 1489 and set(kid_adult[0]) == {"prompt", "kid", "adult"}
assert len(good_bad) == 2226 and set(good_bad[0]) == {"instruction", "chosen", "rejected"}
assert len(public_style) == 50 and {"prompt", "kid", "adult", "fact"} == set(public_style[0])
assert len(public_quality) == 50 and {"prompt", "chosen", "rejected"} == set(public_quality[0])
print({"kid_adult": len(kid_adult), "good_bad": len(good_bad),
       "public_style": len(public_style), "public_quality": len(public_quality)})

In [ ]:
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
TRAIN_MAX_LENGTH = 512
PROMPT_MAX_LENGTH = 256
GEN_MAX_NEW_TOKENS = 160
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

causal_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=LORA_TARGETS,
)

def user_turn(text):
    return [{"role": "user", "content": text}]

def assistant_turn(text):
    return [{"role": "assistant", "content": text}]

def prompt_text(text):
    return tokenizer.apply_chat_template(
        user_turn(text), tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )

def clear_gpu(*objects):
    gc.collect()
    torch.cuda.empty_cache()

def load_causal_for_training():
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.float16,
    )
    model.config.use_cache = False
    return prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

def interval(value, borders):
    letters = ["А", "Б", "В", "Г"]
    return letters[sum(value >= x for x in borders)]

metrics = {}

In [ ]:
def rendered_len(prompt, answer):
    conversation = user_turn(prompt) + assistant_turn(answer)
    return len(tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=False,
        enable_thinking=False,
    ))

style_lengths = [rendered_len(x["prompt"], x["kid"]) for x in kid_adult]
quality_lengths = [max(rendered_len(x["instruction"], x["chosen"]),
                       rendered_len(x["instruction"], x["rejected"])) for x in good_bad]


In [ ]:
with (METRICS_DIR / "style_clf.pkl").open("rb") as f:
    style_bundle = pickle.load(f)

style_vecs = style_bundle["vecs"]
if not isinstance(style_vecs, (list, tuple)):
    style_vecs = [style_vecs]
style_clf = style_bundle["clf"]

def vectorize_style(texts):
    return hstack([vec.transform(texts) for vec in style_vecs], format="csr")

cal_kid = [x["kid"] for x in kid_adult[:200]]
cal_adult = [x["adult"] for x in kid_adult[:200]]
pk = style_clf.predict_proba(vectorize_style(cal_kid)).mean(axis=0)
pa = style_clf.predict_proba(vectorize_style(cal_adult)).mean(axis=0)
simple_col = int(np.argmax(pk - pa))
print("classes:", style_clf.classes_, "simple column:", simple_col,
      "kid/adult calibration:", float(pk[simple_col]), float(pa[simple_col]))

@torch.inference_mode()
def generate_public_style(model, batch_size=4):
    model.eval()
    answers = []
    prompts = [prompt_text(x["prompt"]) for x in public_style]
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    for start in tqdm(range(0, len(prompts), batch_size), desc="greedy generation"):
        batch = tokenizer(
            prompts[start:start + batch_size], return_tensors="pt",
            padding=True, add_special_tokens=False,
        ).to("cuda")
        input_width = batch["input_ids"].shape[1]
        output = model.generate(
            **batch,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        answers.extend(tokenizer.batch_decode(output[:, input_width:], skip_special_tokens=True))
    tokenizer.padding_side = old_padding_side
    return answers

def style_metric(model):
    answers = generate_public_style(model)
    probabilities = style_clf.predict_proba(vectorize_style(answers))[:, simple_col]
    return float(np.mean(probabilities)), answers

задача 1

In [ ]:
sft_dataset = Dataset.from_list([
    {"prompt": user_turn(x["prompt"]), "completion": assistant_turn(x["kid"])}
    for x in kid_adult
])

sft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)
sft_model.config.use_cache = False

sft_args = SFTConfig(
    output_dir=str(RUNS / "sft_trainer"),
    seed=SEED, data_seed=SEED,
    max_length=TRAIN_MAX_LENGTH,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1.0,
    learning_rate=1.5e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True, bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
)
sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=causal_lora,
)
sft_trainer.train()
SFT_DIR = RUNS / "sft_adapter"
sft_trainer.model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)

In [ ]:
sft_trainer.model.config.use_cache = True
metric_1, sft_answers = style_metric(sft_trainer.model)
metrics["1_SFT_P_simple"] = metric_1
print(f" SFT P_simple: {metric_1:.6f}")
print("Интервал:", interval(metric_1, [0.4, 0.7, 0.9]))


In [ ]:
del sft_trainer, sft_model
clear_gpu()

задача 2

In [ ]:
style_dpo_dataset = Dataset.from_list([
    {
        "prompt": user_turn(x["prompt"]),
        "chosen": assistant_turn(x["kid"]),
        "rejected": assistant_turn(x["adult"]),
    }
    for x in kid_adult
])

style_base = load_causal_for_training()
style_model = PeftModel.from_pretrained(
    style_base, SFT_DIR, adapter_name="style_policy", is_trainable=True,
)
style_model.load_adapter(SFT_DIR, adapter_name="style_reference", is_trainable=False)
style_model.set_adapter("style_policy")

style_dpo_args = DPOConfig(
    output_dir=str(RUNS / "style_dpo_trainer"),
    seed=SEED, data_seed=SEED,
    max_length=TRAIN_MAX_LENGTH,
    max_prompt_length=PROMPT_MAX_LENGTH,
    truncation_mode="keep_end",
    beta=0.1,
    loss_type="sigmoid",
    model_adapter_name="style_policy",
    ref_adapter_name="style_reference",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1.0,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True, bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
)
style_dpo_trainer = DPOTrainer(
    model=style_model,
    ref_model=None,
    args=style_dpo_args,
    train_dataset=style_dpo_dataset,
    processing_class=tokenizer,
)
style_dpo_trainer.train()
STYLE_BUNDLE = RUNS / "style_dpo_bundle"
style_dpo_trainer.model.save_pretrained(STYLE_BUNDLE, selected_adapters=["style_policy"])
STYLE_DPO_DIR = STYLE_BUNDLE / "style_policy"


In [ ]:
style_dpo_trainer.model.set_adapter("style_policy")
style_dpo_trainer.model.config.use_cache = True
metric_2, style_dpo_answers = style_metric(style_dpo_trainer.model)
metrics["2_style_DPO_P_simple"] = metric_2
print(f"style-DPO P_simple: {metric_2:.6f}")


In [ ]:

del style_dpo_trainer, style_model, style_base
clear_gpu()

задача 3

In [ ]:
rm_dataset = Dataset.from_list([
    {
        "chosen": user_turn(x["instruction"]) + assistant_turn(x["chosen"]),
        "rejected": user_turn(x["instruction"]) + assistant_turn(x["rejected"]),
    }
    for x in good_bad
])

rm_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)
rm_model.config.pad_token_id = tokenizer.pad_token_id
rm_model.config.use_cache = False

reward_lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    target_modules=LORA_TARGETS,
    modules_to_save=["score"],
)
rm_args = RewardConfig(
    output_dir=str(RUNS / "rm_trainer"),
    seed=SEED, data_seed=SEED,
    max_length=TRAIN_MAX_LENGTH,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1.0,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True, bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
    center_rewards_coefficient=0.01,
)
rm_trainer = RewardTrainer(
    model=rm_model,
    args=rm_args,
    train_dataset=rm_dataset,
    processing_class=tokenizer,
    peft_config=reward_lora,
)
rm_trainer.train()
RM_DIR = RUNS / "reward_adapter"
rm_trainer.model.save_pretrained(RM_DIR)
tokenizer.save_pretrained(RM_DIR)

In [ ]:
@torch.inference_mode()
def reward_scores(model, prompts, answers, batch_size=4):
    model.eval()
    texts = [tokenizer.apply_chat_template(
        user_turn(p) + assistant_turn(a), tokenize=False,
        add_generation_prompt=False, enable_thinking=False,
    ) for p, a in zip(prompts, answers)]
    lengths = [len(tokenizer(t, add_special_tokens=False).input_ids) for t in texts]
    assert max(lengths) <= 768, "Public-пара длиннее лимита reward-метрики"
    result = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[start:start + batch_size], return_tensors="pt", padding=True,
            add_special_tokens=False,
        ).to("cuda")
        result.extend(model(**batch).logits.float().view(-1).cpu().tolist())
    return np.asarray(result)

q_prompts = [x["prompt"] for x in public_quality]
rm_chosen = reward_scores(rm_trainer.model, q_prompts, [x["chosen"] for x in public_quality])
rm_rejected = reward_scores(rm_trainer.model, q_prompts, [x["rejected"] for x in public_quality])
metric_3 = float(np.mean(rm_chosen > rm_rejected))
metrics["3_RM_pairwise_accuracy"] = metric_3
print(f" RM pairwise accuracy: {metric_3:.6f}")


In [ ]:
del rm_trainer, rm_model
clear_gpu()

задача 4

In [ ]:
@torch.inference_mode()
def mean_response_logprobs(model, prompts, answers, batch_size=2):
    model.eval()
    encoded = []
    for p, a in zip(prompts, answers):
        prefix = prompt_text(p)
        prefix_ids = tokenizer(prefix, add_special_tokens=False).input_ids
        answer_ids = tokenizer(a, add_special_tokens=False).input_ids
        full_ids = prefix_ids + answer_ids
        assert len(full_ids) <= 768, "Public-ответ длиннее лимита метрики"
        labels = [-100] * len(prefix_ids) + answer_ids
        encoded.append((full_ids, labels))

    scores = []
    for start in tqdm(range(0, len(encoded), batch_size), desc="mean response logprob"):
        items = encoded[start:start + batch_size]
        width = max(len(ids) for ids, _ in items)
        ids_batch, labels_batch, masks = [], [], []
        for ids, labels in items:
            pad = width - len(ids)
            ids_batch.append(ids + [tokenizer.pad_token_id] * pad)
            labels_batch.append(labels + [-100] * pad)
            masks.append([1] * len(ids) + [0] * pad)
        input_ids = torch.tensor(ids_batch, device="cuda")
        labels = torch.tensor(labels_batch, device="cuda")
        attention_mask = torch.tensor(masks, device="cuda")
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.float()
        token_logp = torch.log_softmax(logits[:, :-1], dim=-1)
        targets = input_ids[:, 1:]
        selected = token_logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        answer_mask = labels[:, 1:].ne(-100)
        batch_scores = (selected * answer_mask).sum(-1) / answer_mask.sum(-1)
        scores.extend(batch_scores.cpu().tolist())
    return np.asarray(scores)

def implicit_preference_accuracy(model):
    prompts = [x["prompt"] for x in public_quality]
    chosen = mean_response_logprobs(model, prompts, [x["chosen"] for x in public_quality])
    rejected = mean_response_logprobs(model, prompts, [x["rejected"] for x in public_quality])
    return float(np.mean(chosen > rejected)), chosen, rejected

In [ ]:
quality_dataset = Dataset.from_list([
    {
        "prompt": user_turn(x["instruction"]),
        "chosen": assistant_turn(x["chosen"]),
        "rejected": assistant_turn(x["rejected"]),
    }
    for x in good_bad
])

quality_base = load_causal_for_training()
quality_model = PeftModel.from_pretrained(
    quality_base, STYLE_DPO_DIR, adapter_name="quality_policy", is_trainable=True,
)
quality_model.load_adapter(STYLE_DPO_DIR, adapter_name="quality_reference", is_trainable=False)
quality_model.set_adapter("quality_policy")

quality_dpo_args = DPOConfig(
    output_dir=str(RUNS / "quality_dpo_trainer"),
    seed=SEED, data_seed=SEED,
    max_length=TRAIN_MAX_LENGTH,
    max_prompt_length=PROMPT_MAX_LENGTH,
    truncation_mode="keep_end",
    beta=0.1,
    loss_type="sigmoid",
    model_adapter_name="quality_policy",
    ref_adapter_name="quality_reference",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1.0,
    learning_rate=3e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True, bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
)
quality_dpo_trainer = DPOTrainer(
    model=quality_model,
    ref_model=None,
    args=quality_dpo_args,
    train_dataset=quality_dataset,
    processing_class=tokenizer,
)
quality_dpo_trainer.train()
QUALITY_DPO_BUNDLE = RUNS / "quality_dpo_bundle"
quality_dpo_trainer.model.save_pretrained(
    QUALITY_DPO_BUNDLE, selected_adapters=["quality_policy"],
)

In [ ]:
quality_dpo_trainer.model.set_adapter("quality_policy")
quality_dpo_trainer.model.config.use_cache = True
metric_4, dpo_chosen, dpo_rejected = implicit_preference_accuracy(quality_dpo_trainer.model)
metrics["4_quality_DPO_implicit_accuracy"] = metric_4
print(f"quality-DPO implicit accuracy: {metric_4:.6f}")


In [ ]:
del quality_dpo_trainer, quality_model, quality_base
clear_gpu()

задача 5

In [ ]:
simpo_base = load_causal_for_training()
simpo_model = PeftModel.from_pretrained(
    simpo_base, STYLE_DPO_DIR, adapter_name="simpo_policy", is_trainable=True,
)
simpo_model.set_adapter("simpo_policy")

simpo_args = CPOConfig(
    output_dir=str(RUNS / "simpo_trainer"),
    seed=SEED, data_seed=SEED,
    max_length=TRAIN_MAX_LENGTH,
    beta=2.0,
    loss_type="simpo",
    simpo_gamma=0.5,
    cpo_alpha=0.0,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1.0,
    learning_rate=3e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True, bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=0,
)
simpo_trainer = CPOTrainer(
    model=simpo_model,
    args=simpo_args,
    train_dataset=quality_dataset,
    processing_class=tokenizer,
)
simpo_trainer.train()
SIMPO_BUNDLE = RUNS / "simpo_bundle"
simpo_trainer.model.save_pretrained(SIMPO_BUNDLE, selected_adapters=["simpo_policy"])

In [ ]:
simpo_trainer.model.set_adapter("simpo_policy")
simpo_trainer.model.config.use_cache = True
metric_5, simpo_chosen, simpo_rejected = implicit_preference_accuracy(simpo_trainer.model)
metrics["5_SimPO_implicit_accuracy"] = metric_5
print(f"SimPO implicit accuracy: {metric_5:.6f}")


итоговые ответы:

In [ ]:
answers = {
    "задача 1": interval(metric_1, [0.4, 0.7, 0.9]),
    "адача 2": interval(metric_2, [0.4, 0.7, 0.9]),
    "адача 3": interval(metric_3, [0.6, 0.75, 0.9]),
    "адача 4": interval(metric_4, [0.6, 0.75, 0.9]),
    "задача 5": interval(metric_5, [0.6, 0.75, 0.9]),
}
for name, value in metrics.items():
    print(f"  {name}: {value:.6f}")
print("Ответы:", answers)